### RAG Pipeline - Data Ingestion to Vector DB

In [6]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\mohab\AppData\Local\Temp\ipykernel_1684\582141364.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
e:\ML-Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read all PDFs inside the directory

In [7]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")

Found 2 PDF files to process

Processing: CFD assignments 1&2.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/NPOPBE+AdvP6975', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 37, '/Type': '/Font', '/Subtype': '/Type1', '/LastChar': 146, '/Widths': [833, 0, 0, 333, 333, 0, 0, 250, 333, 250, 281, 500, 500, 500, 500, 500, 500, 500, 500, 500, 0, 281, 0, 0, 0, 0, 0, 0, 718, 666, 666, 718, 614, 552, 718, 718, 333, 0, 718, 614, 885, 718, 718, 552, 0, 666, 552, 614, 718, 718, 947, 0, 718, 0, 0, 0, 0, 0, 0, 0, 447, 500, 447, 500, 447, 333, 500, 500, 281, 281, 500, 281, 781, 500, 500, 500, 500, 333, 385, 281, 500, 500, 718, 500, 500, 447, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 333, 333], '/FontDescriptor': IndirectObject(13, 0, 2579308598608)}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/NNKGPL+AdvTT5843c571+20', '/Encoding': '

  ✓ Loaded 5 pages

Processing: turbines-compressors-and-fans-by-sm-yahyapdf_compress.pdf
  ✓ Loaded 842 pages

Total documents loaded: 847


In [8]:
all_pdf_documents

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-04-27T13:11:24+00:00', 'source': '..\\data\\pdf\\CFD assignments 1&2.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'CFD assignments 1&2.pdf', 'file_type': 'pdf'}, page_content='Appendix E\nCFD Assignments and\nGuideline for CFD Project\nA sample of three assignments and a guideline for a CFD project clarifying the\naim and objectives are described in this appendix. For students who do not pos-\nsess their own project topics, they are most welcomed to select one from the\nattached project topics (CFD projects A –C) exemplified herein.\nE.1 ASSIGNMENT 1\nE.1.1 Background and Aim\nThe backward-facing step is commonly used as a benchmark for validating\nnumerous flow characteristics, including flow recirculation and reattachment,\nand testing of numerical models and methods. This problem has numerous\napplications in industry, such as for HVAC and combustion chamber.\

### Text splitting into chunks

In [9]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators=[
            "\n\n",
            "\n",
            "",
            " "
        ]
    )
    split_docs = text_splitter.split_documents(documents)
    
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
    

In [10]:
chunks = split_documents(all_pdf_documents)
chunks

Split 847 documents into 1688 chunks

Example chunk:
Content: Appendix E
CFD Assignments and
Guideline for CFD Project
A sample of three assignments and a guideline for a CFD project clarifying the
aim and objectives are described in this appendix. For students ...
Metadata: {'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-04-27T13:11:24+00:00', 'source': '..\\data\\pdf\\CFD assignments 1&2.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'CFD assignments 1&2.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-04-27T13:11:24+00:00', 'source': '..\\data\\pdf\\CFD assignments 1&2.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'CFD assignments 1&2.pdf', 'file_type': 'pdf'}, page_content='Appendix E\nCFD Assignments and\nGuideline for CFD Project\nA sample of three assignments and a guideline for a CFD project clarifying the\naim and objectives are described in this appendix. For students who do not pos-\nsess their own project topics, they are most welcomed to select one from the\nattached project topics (CFD projects A –C) exemplified herein.\nE.1 ASSIGNMENT 1\nE.1.1 Background and Aim\nThe backward-facing step is commonly used as a benchmark for validating\nnumerous flow characteristics, including flow recirculation and reattachment,\nand testing of numerical models and methods. This problem has numerous\napplications in industry, such as for HVAC and combustion chamber.\

### Embedding and VectorStoreDB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


### Embedding

In [12]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        
        try:
            print(f" Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfuly. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
            
    def generate_embeddings(self, texts: List[str]):
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
        
## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager
        

 Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4503.14it/s]


Model loaded successfuly. Embedding dimension: 384


### VectorStore

In [13]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1688


In [14]:
chunks

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-04-27T13:11:24+00:00', 'source': '..\\data\\pdf\\CFD assignments 1&2.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'CFD assignments 1&2.pdf', 'file_type': 'pdf'}, page_content='Appendix E\nCFD Assignments and\nGuideline for CFD Project\nA sample of three assignments and a guideline for a CFD project clarifying the\naim and objectives are described in this appendix. For students who do not pos-\nsess their own project topics, they are most welcomed to select one from the\nattached project topics (CFD projects A –C) exemplified herein.\nE.1 ASSIGNMENT 1\nE.1.1 Background and Aim\nThe backward-facing step is commonly used as a benchmark for validating\nnumerous flow characteristics, including flow recirculation and reattachment,\nand testing of numerical models and methods. This problem has numerous\napplications in industry, such as for HVAC and combustion chamber.\

In [15]:
### Converting text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store un VectorStoreDB
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 1688 texts...


Batches: 100%|██████████| 53/53 [00:55<00:00,  1.05s/it]


Generated embeddings with shape: (1688, 384)
Adding 1688 documents to vector store...
Successfully added 1688 documents to vector store
Total documents in collection: 3376


### Retriever Pipeline

In [16]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vectorstore
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5,score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: {query}")
        print(f"top K: {top_k}, Score Threshold: {score_threshold}")
        
        ## Generate query embeddding
        query_embedding = self.embedding_manager.generate_embeddings(([query]))[0]
        
        ## Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embedding,
                n_results=top_k
            )
            retreieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retreieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1}
                        )
                    print(f"Retrieved {len(retreieved_docs)} documents")
            else:
                
               print("No documents found")
            
            return retreieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
        
        
        
        

In [17]:
rag_retriever

In [18]:
rag_retriever.retrieve("What is turbines")

Retrieving documents for query: What is turbines
top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.62it/s]

Generated embeddings with shape: (1, 384)


Retrieved 1 documents
Retrieved 2 documents
Retrieved 3 documents
Retrieved 4 documents
Retrieved 5 documents


[{'id': 'doc_27e4be65_1573',
  'content': "782 Turbines, Compressors and Fans \n663. Glasgow, J.C. and Robbins, W.H., Utility Operational Experience on \nthe NASAIDOE!Mod-OA 200 kW Wind Turbine, (NASA TM 79084) \nWashington D.C., U.S.A, 1979. \n664. Goiling, G. and Robert, J., 'Tilting at windmills', Electron Power, Vol. \n22, No. 6, pp 347-351, June 1976. \n665. Govinda Raju, S.P. et al., 'Some windmill rotors for use in a rural \nenviron-ment', Rep. FMI, Dept. of Aero. Engg. Indian Institute of \nScience, Bangalore, Jan. 1976. \n666. Hamer, K.l. et al., 'Wind turbine generator control', Brit. Pat. Appl. \n2023237 A, Appl. June 15, 1979. \n667. Herter, E., 'Wind turbine', Brit. Pat. Appl. 2008202A, Appl. Oct. 12, \n1978, Publ. May 1979. \n668. Hinrichsen, E.N., Induction and Synchronous Machines for Vertical Axis \nWind Turbines, Albuquerque, U.S.A, Sandia Labs (Sand 79 7017) June \n1979. \n669. Hunnicutt, C.L. et al., 'An operating 200 kW horizontal axis wind \nturbiny', NASA, TM-790

### Integration between VectorDB context and LLM output

In [ ]:
### RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from pydantic import SecretStr
load_dotenv()

### Initialize Groq LLM (Setting the API Key)

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(temperature=0.1, max_tokens=1024,model = "qwen/qwen3.8-27b", api_key=SecretStr(groq_api_key))

### RAG function

def rag_simple(query, retriever, llm, top_k = 3):
    ### Retrieve the context
    results = retriever.retrieve(query, top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    
    if not context:
        return "No relevant context found to answer this question"
    
    ### Generate the output using the LLM
    prompt = f"""Use the following context to answer the question concisely.
    Context:
    {context}
    
    Question: {query}
    
    Answer:
    
    """
    
    
    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content



    
    
    
    


In [ ]:
answer = rag_simple("eh", rag_retriever,llm)
print(answer)

Retrieving documents for query: What is turbine
top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.13it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents
Retrieved 2 documents
Retrieved 3 documents


A turbine is a power or head-generating machine that employs the dynamic action of a rotating element, the rotor, to change the energy level of a continuously flowing fluid.
